## **Importing Libraries**

In [ ]:
# Importing Libraries
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw, PandasTools, AllChem

import mordred
from mordred import Calculator, descriptors


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings

pd.set_option('display.max_columns', 2000)
warnings.filterwarnings("ignore")

In [ ]:
sns.set(style='whitegrid')

In [ ]:
data = pd.read_csv(r'C:\Users\pc\Desktop\project\tutorial_rdkit\delaney.csv')
data.head()

In [ ]:
mol_list = []

for smile in data['SMILES']:
  mol = Chem.MolFromSmiles(smile)
  mol = Chem.AddHs(mol)
  AllChem.EmbedMolecule(mol)
  mol_list.append(mol)

data = pd.concat([data, pd.DataFrame(mol_list, columns = (['Mol']))], axis=1)

In [ ]:
data.head()

In [ ]:
mol = data['Mol'][54]

In [ ]:
img = Draw.MolToImage(mol)
img

In [ ]:
mols_to_draw = data['Mol'].tolist()[:8]
svg = Draw.MolsToGridImage(
    mols_to_draw,
    molsPerRow=4,
    subImgSize=(200, 200),
    useSVG=True
)
svg_text = svg.data if hasattr(svg, 'data') else svg
with open("mols_grid.svg", "w") as f:
    f.write(svg_text)

print("tsayft: mols_grid.svg")
svg  

In [ ]:
# Creating a descriptor calculator with all descriptors
calc = Calculator(descriptors, ignore_3D=False)

all_desc = calc.pandas(data['Mol'])

In [ ]:
all_desc.head()

In [ ]:
all_desc.shape

In [ ]:
data.head()

In [ ]:
df_index = data[['Compound ID', 'SMILES', 'measured log(solubility:mol/L)']]

In [ ]:
df = pd.concat([df_index, all_desc], axis=1)
df.head()

In [ ]:
df.to_excel('delaney_mordred.xlsx', index=None)

## **Loading the Dataset**

In [ ]:
df = pd.read_excel(r'C:\Users\pc\Desktop\project\tutorial_rdkit\delaney_mordred.xlsx')


## **Data Preprocessing**



1.   Removing missing values/non-numerical values
2.   Remove constant values
3.   Remove highly correlated values





In [ ]:
df.head()

In [ ]:
data = df.iloc[:,3:]

In [ ]:
data.head()

In [ ]:
data.isnull().sum().sum()

In [ ]:
column_num = []
column_bool = []
for column in data.columns:
  column_type = data[column].dtype
  if column_type == 'object':
      pass
  elif column_type =='bool':
      column_bool.append(column)
  else:
      column_num.append(column)

In [ ]:
len(column_num)

In [ ]:
column_bool

In [ ]:
data['GhoseFilter'].unique()

In [ ]:
gf = data['GhoseFilter'].astype(int)
gf.value_counts().plot(kind='bar')


In [ ]:
data = data[column_num + column_bool]

In [ ]:
data.shape

In [ ]:
def remove_constant_values(data):
    return [e for e in data.columns if data[e].nunique() == 1]

drop_col = remove_constant_values(data)
#drop_col

new_df_columns = [e for e in data.columns if e not in drop_col]
new_df = data[new_df_columns]
new_df

In [ ]:
len(drop_col)

In [ ]:

def correlation(dataset, threshold):
    col_corr = set()  
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold: 
                colname = corr_matrix.columns[i]  
                col_corr.add(colname)
    return col_corr

In [ ]:

corr_features = correlation(new_df, 0.80)
print("No. of features to drop : ",len(set(corr_features)))

new_df.drop(corr_features,axis=1,inplace=True)

In [ ]:
new_df.shape

In [ ]:
new_df.head()

In [ ]:
new_df['Lipinski'] = df["Lipinski"].astype(int)
new_df['GhoseFilter'] = df["GhoseFilter"].astype(int)

In [ ]:
new_df.head()

In [ ]:
df_final = pd.concat([df.iloc[:, :3], new_df], axis=1)
df_final.head()

In [ ]:
df_final.shape

In [ ]:
df_final.to_csv('delaney_mordred_truncated.csv', index=None)

## **Data Analysis**

In [ ]:
df_final['measured log(solubility:mol/L)'].describe()

In [ ]:
plt.hist(df_final['measured log(solubility:mol/L)'])

In [ ]:
df_final = df_final.iloc[:, 2:]

In [ ]:
corr = df_final.corr()
corr

In [ ]:
corr_sorted = abs(corr[['measured log(solubility:mol/L)']]).sort_values(by ='measured log(solubility:mol/L)', ascending=False)
corr_sorted = corr_sorted.iloc[1:5, :]
corr_sorted.rename(columns={'measured log(solubility:mol/L)' : 'correlation_coef'}, inplace=True)
corr_sorted


In [ ]:
fig = plt.figure(1, figsize=(6,6))
ax1 = fig.add_subplot(111)
plt.bar(x = corr_sorted.index, height = corr_sorted['correlation_coef'], color = 'green')
ax1.set_xlabel('Top Correlated Descriptors', weight='bold')
ax1.set_ylabel('Correlation Coefficient', weight='bold')

In [ ]:
fig = plt.figure(4, figsize=(10,10))
ax = fig.add_subplot(221)
plt.scatter(x = df_final['measured log(solubility:mol/L)'], y = df_final['FilterItLogS'], color = 'green')
ax.set_xlabel('Log Solubilities', weight='bold')
ax.set_ylabel('FilterItLogS', weight='bold')

ax = fig.add_subplot(222)
plt.scatter(x = df_final['measured log(solubility:mol/L)'], y = df_final['PEOE_VSA6'], color = 'green')
ax.set_xlabel('Log Solubilities', weight='bold')
ax.set_ylabel('PEOE_VSA6', weight='bold')

ax = fig.add_subplot(223)
plt.scatter(x = df_final['measured log(solubility:mol/L)'], y = df_final['RNCG'], color = 'green')
ax.set_xlabel('Log Solubilities', weight='bold')
ax.set_ylabel('RNCG', weight='bold')

ax = fig.add_subplot(224)
plt.scatter(x = df_final['measured log(solubility:mol/L)'], y = df_final['ABC'], color = 'green')
ax.set_xlabel('Log Solubilities', weight='bold')
ax.set_ylabel('ABC', weight='bold')
plt.tight_layout()

In [ ]:
df_final = pd.read_csv(r"C:\Users\pc\Desktop\project\tutorial_rdkit\delaney_mordred_truncated.csv")

## **Building Machine Learning Model**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import logistic

In [ ]:
df_final = pd.read_csv(r"C:\Users\pc\Desktop\project\tutorial_rdkit\delaney_mordred_truncated.csv")


In [ ]:
df_final.head()

In [ ]:
df_final.shape

In [ ]:
y = df_final['measured log(solubility:mol/L)']

In [ ]:
scaled_DF = pd.DataFrame(StandardScaler().fit_transform(df_final.iloc[:,3:]))
scaled_DF.columns = df_final.iloc[: , 3:].columns


**Standerscaler for how the features away from the mean**

In [ ]:
scaled_DF

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(scaled_DF, y, test_size=0.20, random_state=45)

In [ ]:
X_test.shape

**$$y = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + ... + \beta_p x_p$$**
sm.statsmodel is used to be beta_0 diffrent to 0 in this equation system(Linear_regression)


In [ ]:

x = sm.add_constant(X_train)
results = sm.OLS(y_train, x).fit()
print(results.summary())

In [ ]:
# multiple linear regression model using scikitlearn
lr = LinearRegression()
lr.fit(X_train, y_train)

**clear sign of overfiting is the diminution in the score test by 33%**

In [ ]:
print(f'The r2 score for train set is : {lr.score(X_train, y_train)}')
print(f'The r2 score for test set is : {lr.score(X_test, y_test)}')

In [ ]:
y_hat_train = lr.predict(X_train)
y_hat_test = lr.predict(X_test)

In [ ]:
r2_score(y_train, y_hat_train)

In [ ]:
r2_score(y_test, y_hat_test)

In [ ]:
y_hat_train.shape

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

In [ ]:
print(f'The r2 score for train set is : {rf.score(X_train, y_train)}')
print(f'The r2 score for test set is : {rf.score(X_test, y_test)}')

## **Cross-Validation**

In [ ]:
val = cross_val_score(rf, X_train, y_train,  scoring='r2', cv=5)

In [ ]:
val.mean()

## **Visualizing the Results**

In [ ]:
plt.scatter(y_test, rf.predict(X_test), label='Test')
sns.regplot(x = y_train, y = rf.predict(X_train), color = 'maroon',scatter_kws={'alpha':0.3}, label='Train' )
plt.xlabel('Measured Solubilities')
plt.ylabel('Predicted Solubilites')
plt.legend();

## **Dimensionality Reduction Techniques**

# **Principal Component Analysis**

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
scaled_DF.head()

In [ ]:
scaled_DF.shape


**PCA is a dimensionality reduction technique that transforms correlated features into a smaller set of linearly uncorrelated variables called principal components.**

In [ ]:
pca = PCA(n_components=8)
pc = pca.fit_transform(scaled_DF)
pc_df = pd.DataFrame(data = pc)

In [ ]:
pc_df.head()

In [ ]:
pca.explained_variance_ratio_

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(pc_df, y, test_size=0.20, random_state=45)

In [ ]:
x = sm.add_constant(X_train)
results = sm.OLS(y_train, x).fit()
print(results.summary())

In [ ]:
lr = LinearRegression()
lr.fit(X_train, y_train)

print(f'The r2 score for train set is : {lr.score(X_train, y_train)}')
print(f'The r2 score for test set is : {lr.score(X_test, y_test)}')

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
print(f'The r2 score for train set is : {rf.score(X_train, y_train)}')
print(f'The r2 score for test set is : {rf.score(X_test, y_test)}')

# **t-SNE**

t-sne for reduce the nb of features and plot the features by importing just the important features  TSNE(n_components=2) who affected so much the output 


In [ ]:
from sklearn.manifold import TSNE

In [ ]:
tsne = TSNE(n_components=2) 
ts = tsne.fit_transform(scaled_DF)
tsne_df = pd.DataFrame(data = ts)


In [ ]:
tsne_df.head()

**Visuaization in Lower Dimensions**


we have to separate the data target a tow clusters 
0 : less solubulity 
1 : high solubility 


In [ ]:
y.median()

In [ ]:
sol_cls = [int(boolean) for boolean in list(map(lambda s: s>-2.8, y))]


In [ ]:
from collections import Counter
Counter(sol_cls)

In [ ]:
plt.figure(figsize=(8,6))
scatter = plt.scatter(tsne_df.iloc[:,0],tsne_df.iloc[:,1],
                      c = sol_cls,cmap='plasma'
                      
                      )
plt.legend(handles=scatter.legend_elements()[0], labels=['More Soluble', 'Less Soluble'], loc = 1)
plt.xlabel('tsne Component 1')
plt.ylabel('tsne Component 2')

**for fox the overfiting probleme and know where the faetures who affect the less and the high soluility**


In [ ]:
df_final.head()


In [ ]:

y = df_final['measured log(solubility:mol/L)']

In [ ]:

scaled_DF = pd.DataFrame(StandardScaler().fit_transform(df_final.iloc[:,3:]) , columns = df_final.iloc[: , 3:].columns)


In [ ]:
scaled_DF

In [ ]:
X_train , X_test , y_train , y_test = train_test_split(scaled_DF, y , test_size = 0.2 , random_state = 45)


In [ ]:

rf = RandomForestRegressor()
rf.fit(X_train, y_train)

In [ ]:

print(f'The r2 score for train set is : {rf.score(X_train, y_train)}')
print(f'The r2 score for test set is : {rf.score(X_test, y_test)}')

**showing the important features only maje the model powerful**

In [ ]:

scaled_DF = rf.feature_importances_

dicts = {
    'Features':[x for x in df_final.iloc[:,3:].columns],
    'Importance':importance
    }
DF_imp = pd.DataFrame(dicts)
DF_imp = DF_imp.sort_values('Importance',ascending=False)
DF_imp.to_excel('imp.xlsx', index=None)


**visualisation the imporatant features**

In [ ]:
top_desc_fi = DF_imp[:6]
plt.subplots(figsize=(6,6))
sns.barplot(data = top_desc_fi , x ='Features'  , y ='Importance' )
plt.xlabel('the features')
plt.ylabel('the importance')



**Scikit Learn's Permutation Importance**

Permutation importance works by permuting the values of a single feature and measuring the change in the model's performance (e.g., accuracy or mean squared error)

In [ ]:
from sklearn.inspection import permutation_importance


In [ ]:
result = permutation_importance(
    rf, X_test, y_test, random_state=42)

dicts = {
    'Features':[x for x in df_final.iloc[:,3:].columns],
    'Importance':result.importances_mean
}
DF_pi = pd.DataFrame(dicts)
DF_pi = DF_pi.sort_values('Importance',ascending=False) 

DF_pi.to_excel('imp.xlsx', index=None)

top_desc_pi = DF_pi[:5]
sns.barplot(data=top_desc_pi, x = 'Features', y='Importance', palette = 'Set2')

plt.xticks(rotation = 90)
plt.show()

In [ ]:

top_desc_pi = DF_pi[:5]
sns.barplot(data=top_desc_pi, x = 'Features', y='Importance', palette = 'Set2')

plt.xticks(rotation = 90)

plt.show()

In [ ]:


top_desc_pi.head()



**SHAP : Measures exactly how much each feature contributed to the final output.**

In [ ]:
import shap

In [ ]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values , X_test)



In [ ]:
top_desc_pi['Features'][:5]

In [ ]:
scaled_df_5 = scaled_DF[top_desc_fi['Features'][:5]]
scaled_df_5.head() 

In [ ]:
y
X_train , X_test , y_train , y_test = train_test_split(scaled_df_5 , y , test_size = 0.2 , random_state = 45)
lr = LinearRegression()
lr.fit(X_train, y_train)
print(f'The r2 score for train set is : {lr.score(X_train, y_train)}')
print(f'The r2 score for test set is : {lr.score(X_test, y_test)}')
rf = RandomForestRegressor()
rf.fit(X_train, y_train)
y_hatrf_train = rf.predict(X_train)
y_hatrf_test = rf.predict(X_test)


print(f'The r2 score for train set is : {rf.score(X_train, y_train)}')
print(f'The r2 score for test set is : {rf.score(X_test, y_test)}')
